# DR-TMLE: an interval when the recorded assignment rule is hard to model

This notebook fits DR-TMLE, a TMLE variant that solves two extra score equations. The extra
equations can protect the interval when one nuisance model converges to the wrong limit. That
protection needs rate conditions on the other fits. Each step shows its code, its output, and what
the output tells you. The [DR-TMLE reference](../technical-reference/dr-tmle/index.md) holds the theorem, the refusals, and the
release claim. If TMLE is new to you, start with [point-treatment TMLE](point-treatment-tmle.ipynb).

## The applied question

The navigation program repeats its evaluation of the offer with 2,000 discharges. The program
logged every common cause of assignment and outcome. The recorded assignment rule has a squared
term, an interaction, and a threshold. A main-effects logistic model of that rule is therefore
misspecified.

The program analyst asks one question. Can the program report an interval for the average treatment
effect (ATE) when the assignment model is the doubtful one? An unrecorded common cause is a
different failure, and DR-TMLE does not repair it.

## What you will learn

| after this notebook you can | the step that shows it |
| --- | --- |
| say which nuisance failure DR-TMLE addresses, and which it does not | "Why this method" |
| explain why an unadjusted association is not the ATE | Step 3 |
| check that DR-TMLE is available for your estimand | Step 5 |
| configure the reduced regressions and the guard | Step 6 |
| confirm that an empty guard reproduces the ordinary TMLE | Step 7 |
| explain why solved score equations do not certify the fits | Step 8 |
| read the correction report and the reduced-regression diagnostics | Step 9 |
| say which assumption DR-TMLE does not relax, and why `cleverly` refuses the omitted-variable bounds here | Step 10 |
| state the answer to the program's question and its condition | "How far to trust this" |

## Why this method

An ordinary TMLE stays consistent when one nuisance is consistent. Its interval needs both
nuisances to converge fast enough. The table compares the two estimators for the case this page
is about.

| estimator | when the assignment model converges to the wrong limit |
| --- | --- |
| ordinary TMLE | stays consistent. If the outcome fit converges too slowly, the remainder can dominate the root-n scale. The usual influence-curve interval then need not attain nominal coverage |
| DR-TMLE | solves two extra score equations built from reduced-dimension regressions. Under Theorem 1 of Benkeser et al. (2017), it stays asymptotically linear, given [rate conditions](../technical-reference/dr-tmle/theorem.md#the-remainder-terms-and-the-rate-conditions) on the outcome fit and the reduced regressions |

When both nuisances are consistent, the corrections converge to zero. DR-TMLE then has no
asymptotic advantage and adds finite-sample cost.
[What this solves](../technical-reference/dr-tmle/index.md#what-this-solves) gives the remainder
argument.

| term | plain meaning |
| --- | --- |
| estimand | the number the question asks for, written before any model is chosen. See [estimands](../user-guide/estimands.md) |
| nuisance | a model the estimate needs but the question does not ask about. Here, the outcome regression Q and the treatment mechanism g. See [point-treatment TMLE](../technical-reference/point-treatment-tmle.md) |
| double robustness | the point estimate stays consistent when either nuisance model is consistent. See [point-treatment TMLE](../technical-reference/point-treatment-tmle.md) |
| targeting | a small update to Q, weighted by g, that removes first-order bias. See [targeting and bounds](../user-guide/methods-learners.md#targeting-and-bounds) |
| influence curve | how much each row moves the estimate. Its variance gives the standard error. See [inference](../technical-reference/inference.md) |
| cross-fitting | each row's nuisance prediction comes from models fit without that row. See [CV-TMLE](../technical-reference/cv-tmle.md) |
| reduced regression | a regression with one input, the fitted value of the other nuisance. DR-TMLE fits three. See [the algorithm](../technical-reference/dr-tmle/index.md#the-algorithm-as-implemented) |
| score equation | a mean over the rows that targeting drives to zero. See [DR-TMLE diagnostics](../technical-reference/dr-tmle/diagnostics.md) |


## Step 1: set up

The setup imports the learners and prints the installed `cleverly` version. Every fit below passes
its learners, fold count, and random seed explicitly, so a rerun reproduces the stored outputs.


In [1]:
from dataclasses import replace

import pandas as pd
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.2


**What this output tells you.** The stored outputs in this notebook came from the version named
above. A different version can print different numbers.


## Step 2: the data

The data come from `navigation_data`. It draws `make_nonlinear_bounded`, the synthetic law of the
point-treatment tutorial, under the program's column names. This page uses a new size and seed. The
code prints the column names, the first rows, the share of discharges offered navigation, and the
true values of the law.


In [2]:
from cleverly.datasets import navigation_data

frame, truth = navigation_data(n=2_000, seed=55)
print("rows and columns:", frame.shape)
print("columns:", list(frame.columns))
print(frame.head().round(3))
print()
print("share offered navigation:", round(float(frame["transition_navigation"].mean()), 3))
print("known values of the synthetic law:")
for key in ("ey1", "ey0", "ate"):
    print(f"  {key}: {truth[key]:.3f}")

rows and columns: (2000, 6)
columns: ['transition_score', 'transition_navigation', 'discharge_risk', 'prior_utilization', 'medication_burden', 'age']
   transition_score  transition_navigation  discharge_risk  prior_utilization  medication_burden    age
0             0.737                    0.0           0.842             -2.976             -0.305  1.450
1             0.256                    1.0          -1.244              0.053              1.500 -1.168
2             0.826                    0.0           0.811              1.899              0.447  1.629
3             0.710                    1.0          -0.138             -0.421              0.466 -1.440
4             0.792                    1.0           1.110              0.045             -1.293  1.132

share offered navigation: 0.45
known values of the synthetic law:
  ey1: 0.567
  ey0: 0.404
  ate: 0.163


**What this output tells you.** The frame has 2,000 discharges and the six program columns. The
share offered navigation is 0.45. The true arm means are 0.567 with the offer and 0.404 without it,
so the true ATE is 0.163.

| feature of the law | what it means for this page |
| --- | --- |
| the true propensity has a squared term, an interaction, and a threshold | a main-effects logistic model of assignment is misspecified. It converges to the wrong limit |
| the true outcome mean is nonlinear | gradient boosting is flexible enough for it, so this page treats the outcome fit as the credible nuisance |
| the score is a share of the maximum score | its support is known, so the cross-fitted fit in Step 6 declares `q_bounds` |
| the four baseline covariates are standardized (mean 0, SD 1) | a negative value is below the average |

The generator `nonlinear_bounded_dgp` in `cleverly.datasets` defines both functions. A real program
has no `truth`, and a real analyst does not know which nuisance is misspecified. Every comparison
against the truth below is a teaching device.


## Step 3: association first

The code compares the two treatment arms before adjustment. It prints each arm's mean score
and baseline covariates beside the unadjusted difference and the true ATE.

In [3]:
covariates = ["discharge_risk", "prior_utilization", "medication_burden", "age"]
by_arm = frame.groupby("transition_navigation")[["transition_score", *covariates]].mean()
print(by_arm.round(3))
print()
unadjusted = by_arm.loc[1.0, "transition_score"] - by_arm.loc[0.0, "transition_score"]
print(f"unadjusted difference in mean score: {unadjusted:.3f}")
print(f"population ATE:                      {truth['ate']:.3f}")

                       transition_score  discharge_risk  prior_utilization  medication_burden    age
transition_navigation                                                                               
0.0                               0.403          -0.209             -0.000              0.045 -0.041
1.0                               0.601           0.304             -0.009              0.005  0.026

unadjusted difference in mean score: 0.197
population ATE:                      0.163


**What this output tells you.** The offered group scores 0.197 higher before adjustment.
The true ATE is 0.163. The mean `discharge_risk` is 0.304 in the offered group and -0.209 in
the usual-support group.

The arms differ on a baseline variable that affects assignment and the outcome. The unadjusted
difference therefore does not answer the ATE question. DR-TMLE changes the interval's nuisance
conditions. It does not remove the need to adjust for measured common causes.


## Step 4: write the protocol

A `StudyProtocol` records the scientific design before any model runs. Every result fitted from it
carries its fingerprint. This page starts from `navigation_protocol()`, the protocol of the
[shared study design](index.md#the-shared-study-design). `dataclasses.replace` changes only the
first assumption rationale. [Point-treatment TMLE](point-treatment-tmle.ipynb) explains each field.

In [4]:
from cleverly.datasets import navigation_protocol

program = navigation_protocol()
protocol = replace(
    program,
    assumption_rationale=(
        "The program logged every input of the recorded assignment rule, "
        "and the baseline variables cover the measured common causes",
        *program.assumption_rationale[1:],
    ),
)
print("\n".join(protocol.summary_lines()))

causal study protocol: schema 1; ffaef567af2a6440
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measurement and before the navigation offer
treatment strategies: ['Offer standard transition navigation', 'Provide usual discharge support']
treatment versions: ['Bedside transition plan and two scheduled navigator contacts within 30 days', 'No access to the transition-navigation offer']
outcome: Patient-reported transition score, as a share of the maximum score
horizon: 30 days after discharge
intercurrent-event handling: ['Use the transition score regardless of readmission', 'Analyze the offer regardless of completed contacts', 'The protocol scores death before day 30 as the worst transition score (composite strategy)']
interference unit: Individual patient
assumption rationale: ['The p

**What this output tells you.** The first line gives the schema version and the fingerprint
`ffaef567af2a6440`. The fields match the point-treatment protocol except one. The changed field is
`assumption rationale`, and only its first entry changes. That entry records that the program logged
every input of the recorded assignment rule.

No protocol field records which nuisance model the analyst doubts. That doubt is an analysis
choice, not a design element. The `DRTMLEMethod` in Step 6 owns it through `guard=`, and the typed
estimand in Step 5 owns the contrast.


## Step 5: design and identification

DR-TMLE targets the same parameter as the ordinary TMLE, under the same assumptions. The design
keeps every common cause in the adjustment set. The code identifies the ATE and prints the method
catalog for it. It then asks the catalog about the average treatment effect on the treated (ATT).


In [5]:
from cleverly import ATE, ATT, CausalStudy, PointTreatment

study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=tuple(covariates),
    ),
    protocol=protocol,
)
effect = study.identify(ATE(reference=0))
print(effect.summary())

print()
print("methods for the ATE:")
for method in effect.available_methods():
    print(f"  {method.name}: {method.available}")
att_catalog = {
    method.name: method for method in study.identify(ATT(reference=0)).available_methods()
}
print("drtmle for the ATT:", att_catalog["drtmle"].available, "-", att_catalog["drtmle"].reason)

average treatment effect, E[Y^a] - E[Y^reference]
identified by explicit-adjustment: E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
adjustment/history: ['discharge_risk', 'prior_utilization', 'medication_burden', 'age']
required nuisances: ['outcome_regression', 'treatment_mechanism']
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; ffaef567af2a6440
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measu

**What this output tells you.** The summary gives the observed-data formula, the two required
nuisances, and four assumptions. It then repeats the stored protocol. The catalog lists `drtmle`
as available for this ATE. For the ATT it prints `False` and the reason.

| catalog outcome | when it happens |
| --- | --- |
| available | a point-treatment arm contrast, such as this ATE |
| unavailable in the catalog | an `ATT`, `ATC`, MSM, or intervention-axis effect. Selecting `drtmle` raises before any nuisance is fitted |
| refused at fit time | the other refusals, such as composition with collaborative TMLE. [Refused by name](../technical-reference/dr-tmle/supported-estimands.md#refused-by-name) lists them |

DR-TMLE relaxes none of the four assumptions. It changes only the nuisance conditions that the
interval needs.


## Step 6: estimate with DR-TMLE

The primary nuisances are the analyst's: a flexible outcome regression and a crude assignment
model. Each reduced regression has one input, so a spline can fit it fast. Each reduction below is
a Super Learner over a linear and a spline candidate. The
[`drtmle` vignette](https://github.com/benkeser/drtmle/blob/538a3a264c1ca984b6d88978ca7f96165f43152c/vignettes/using_drtmle.Rmd)
uses the same pairing, `SL.glm` and `SL.gam`.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | gradient boosting | fits Q, the expected score given the offer and the covariates |
| `treatment_learner` | main-effects logistic regression | fits g. On this law it is misspecified by construction |
| `CrossFitting(n_folds=3)` | three folds | predicts each row from models that did not see that row. The default reduced cross-fitting refuses fewer than three folds |
| `Targeting(q_bounds=(0.0, 1.0))` | the declared score support | fixes the outcome scale before any fold is drawn |
| `reduced_outcome_learner` | Super Learner, linear and spline | fits the conditional means $Q_r$ and $g_{r2}$ |
| `reduced_treatment_learner` | Super Learner, logistic and spline | fits the probability $g_{r1}$ |
| `guard` | the default, `("Q", "g")` | poses both extra equations. `guard=("g",)` poses only the equation against a wrong assignment model |
| `Runtime(random_state=55, n_jobs=1)` | fixed seed, one process | makes the fit reproducible |

DR-TMLE cross-fits its primary nuisances, so the declared support is not optional here. With
`q_bounds=None` the fit would read the outcome scale from the held-out rows as well, and `cleverly`
refuses that fit. The ordinary TMLE in Step 7 declares the same support, so the comparison changes
only the guard.

The guard names the nuisance you doubt, not the fit that its equation updates. The next table
defines the three reduced regressions for each arm $a$. In it, $1_a$ is 1 for a row in arm $a$ and
0 otherwise. Step 9 prints their diagnostics under the output keys.

| output key | reduced regression | its one input | the guard that uses it |
| --- | --- | --- | --- |
| `qr` ($Q_r$) | the outcome residual $Y - \hat Q$ in arm $a$ | the fitted assignment probability $\hat g$ | `"Q"` |
| `gr1` ($g_{r1}$) | the arm indicator $1_a$ | the fitted outcome $\hat Q$ | `"g"` |
| `gr2` ($g_{r2}$) | $(1_a - \hat g)/\hat g$ | the fitted outcome $\hat Q$ | `"g"` |

[The algorithm](../technical-reference/dr-tmle/index.md#the-algorithm-as-implemented) gives the three equations and the corrected
influence curve for each guard.


In [6]:
from cleverly import CrossFitting, DRTMLEMethod, ModelSpec, Runtime, SuperLearner, Targeting

models = ModelSpec(
    outcome_learner=HistGradientBoostingRegressor(random_state=55),
    treatment_learner=LogisticRegression(max_iter=1000, random_state=55),
)
folds = CrossFitting(n_folds=3)
declared_support = Targeting(q_bounds=(0.0, 1.0))
runtime = Runtime(random_state=55, n_jobs=1)


def spline(final):
    return make_pipeline(SplineTransformer(n_knots=5, knots="quantile"), final)


reduced_outcome = SuperLearner(
    library=[
        ("linear", LinearRegression(n_jobs=1)),
        ("spline", spline(LinearRegression(n_jobs=1))),
    ],
    task="regression",
    n_folds=3,
    random_state=55,
    n_jobs=1,
)
reduced_treatment = SuperLearner(
    library=[
        ("logistic", LogisticRegression(max_iter=1000, random_state=55)),
        ("spline", spline(LogisticRegression(max_iter=1000, random_state=55))),
    ],
    task="classification",
    n_folds=3,
    random_state=55,
    n_jobs=1,
)
drtmle = DRTMLEMethod(
    models=models,
    cross_fitting=folds,
    targeting=declared_support,
    runtime=runtime,
    reduced_outcome_learner=reduced_outcome,
    reduced_treatment_learner=reduced_treatment,
)
guarded = effect.estimate(method=drtmle)
print(guarded.summary())
print()
print("guard:", drtmle.guard, "| reduction:", drtmle.reduction)
point = guarded["ate"]
print(f"estimate:        {point.psi:.5f}")
print(f"standard error:  {point.std_error:.5f}")
print(f"95% CI:          ({point.ci[0]:.5f}, {point.ci[1]:.5f})")
print(f"population ATE:  {truth['ate']:.3f}")

Targeted maximum likelihood estimation
n = 2000; covariates = 4; P(A=1) = 0.4495
causal estimand: average treatment effect, E[Y^a] - E[Y^reference]
identification: explicit-adjustment; E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
required nuisances: outcome_regression, treatment_mechanism
identification assumptions: consistency: Y = Y^a when A = a; no interference: one unit's potential outcome does not depend on other units' treatment assignments; no unmeasured confounding: Y^a is independent of A given W; positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; ffaef567af2a6440
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseli

**What this output tells you.** The summary does not name DR-TMLE. Its header is the ordinary
TMLE header. The line `stacked CV-TMLE (Levy)` names how the primary nuisances are cross-fitted over
3 folds, which [CV-TMLE](../technical-reference/cv-tmle.md) explains. The summary shows the
propensity truncation bound, [0.01471, 0.9853]. The line after the summary confirms the default
guard and the univariate reduction.

The summary prints no `outcome scaled` line. The declared support already matches the score's own
scale, so the fit rescales nothing and updates the score with a logistic fluctuation on that scale.

The estimate is 0.16465 with a standard error of 0.00682. The 95% interval
(0.15129, 0.17801) contains the true ATE of 0.163. That is one draw, not a coverage result. The
summary does not print the reduced regressions or the corrections. Steps 8 and 9 read them.


## Step 7: compare with the ordinary TMLE

An empty guard solves no extra equation, so it must reproduce the ordinary TMLE exactly. The code
fits the ordinary TMLE with the same learners, folds, declared support, and seed. It fits DR-TMLE
with `guard=()` and compares the two estimates. It then prints the ordinary fit beside the guarded
fit.


In [7]:
from cleverly import TMLEMethod

ordinary = effect.estimate(
    method=TMLEMethod(
        models=models,
        cross_fitting=folds,
        targeting=declared_support,
        runtime=runtime,
    )
)
empty_guard = effect.estimate(method=replace(drtmle, guard=()))
print("empty guard reproduces the ordinary TMLE:", ordinary["ate"].psi == empty_guard["ate"].psi)
print()
for label, fitted in (("ordinary TMLE", ordinary), ("DR-TMLE", guarded)):
    point = fitted["ate"]
    low, high = point.ci
    print(f"{label:14s} psi={point.psi:.5f}  se={point.std_error:.5f}  CI=({low:.5f}, {high:.5f})")
print(f"population ATE: {truth['ate']:.3f}")
shift = (guarded["ate"].psi - ordinary["ate"].psi) / ordinary["ate"].std_error
se_ratio = guarded["ate"].std_error / ordinary["ate"].std_error
print(f"DR-TMLE shift, in ordinary standard errors: {shift:+.2f}")
print(f"standard-error ratio, DR-TMLE / ordinary:   {se_ratio:.2f}")

empty guard reproduces the ordinary TMLE: True

ordinary TMLE  psi=0.16606  se=0.00691  CI=(0.15251, 0.17960)
DR-TMLE        psi=0.16465  se=0.00682  CI=(0.15129, 0.17801)
population ATE: 0.163
DR-TMLE shift, in ordinary standard errors: -0.20
standard-error ratio, DR-TMLE / ordinary:   0.99


**What this output tells you.** The first line prints `True`. The empty-guard estimate equals the
ordinary estimate exactly. The equality fixes what the variant is: the same estimator plus extra
equations, not a different target.

| quantity | ordinary TMLE | DR-TMLE |
| --- | --- | --- |
| estimate | 0.16606 | 0.16465 |
| standard error | 0.00691 | 0.00682 |
| 95% interval | (0.15251, 0.17960) | (0.15129, 0.17801) |

The guarded estimate moves because the extra fluctuations change the targeted fits. On this draw
it moves by -0.20 ordinary standard errors, less than one. The standard-error ratio is 0.99. That
resemblance says nothing about either interval's coverage.


## Step 8: the failure mode, solved scores do not certify the fits

Score equations describe the targeting step. They do not measure how far a fitted function is from
the true one. The code refits DR-TMLE with constant reductions. A constant reduction ignores its
input, so it cannot follow any dependence on the fitted nuisance. The code prints each fit's
estimate and its two score checks. It then prints the ordinary TMLE of Step 7 for comparison.

In [8]:
crude = replace(
    drtmle,
    reduced_outcome_learner=DummyRegressor(),
    reduced_treatment_learner=DummyClassifier(strategy="prior"),
)
crude_fit = effect.estimate(method=crude)
for label, fitted in (("spline reductions", guarded), ("constant reductions", crude_fit)):
    point = fitted["ate"]
    scores = fitted.diagnostics.score_equations()
    corrections = fitted.diagnostics.corrections()
    print(
        f"{label:20s} psi={point.psi:.5f}  se={point.std_error:.5f}  "
        f"score equations passed={scores.passed}  corrections passed={corrections.passed}"
    )
point = ordinary["ate"]
print(f"{'ordinary TMLE':20s} psi={point.psi:.5f}  se={point.std_error:.5f}")

spline reductions    psi=0.16465  se=0.00682  score equations passed=True  corrections passed=True
constant reductions  psi=0.16568  se=0.00690  score equations passed=True  corrections passed=True
ordinary TMLE        psi=0.16606  se=0.00691


**What this output tells you.** Both DR-TMLE fits print `passed=True` for the score equations
and for the corrections. The spline fit estimates 0.16465. The constant fit estimates 0.16568, with
a standard error of 0.00690. The checks cannot tell the two fits apart.

The last line is the ordinary TMLE, which estimates 0.16606 with a standard error of 0.00691. The
constant fit is close to it on this draw, so the constant reductions change little. The checks
still pass, because targeting solves whichever equations the reductions pose. A passing check
therefore does not show that the extra equations protect the interval.

On this draw the spline fit lands nearer the true ATE of 0.163 than the constant fit does. That
ordering is not evidence either. One draw cannot rank two reductions, and a real analysis has no
truth to compare against.

[Solved scores do not establish nuisance consistency](../technical-reference/dr-tmle/diagnostics.md#solved-scores-do-not-establish-nuisance-consistency)
gives the exact-law test behind this rule. In that test, wrong reductions move the estimate, and
every score equation still passes.


## Step 9: diagnostics, what the fit can show

The combined assessment presents validation, diagnostics, and sensitivity together. The code
prints its summary and three retained outputs. They are the correction check, the nuisance report,
and the reduced-regression diagnostics. `guarded.extra["drtmle"].diagnostics` holds one Super
Learner record per arm and fold for each reduced regression.


In [9]:
assessment = guarded.assess()
print(assessment.summary())
print("needs attention:", tuple(item.name for item in assessment.attention))
print()
print(assessment.report("corrections").summary())
print()
print(assessment.report("nuisance_models").summary())
print()
reduced = guarded.extra["drtmle"].diagnostics
for family, fits in reduced.items():
    print(family, "best candidate per arm and fold:", [fit.best for fit in fits])

Returned results
----------------
surface      operation        result                                                                        
-----------  ---------------  ------------------------------------------------------------------------------
validation   support          maximum truncated fraction 0.0%; minimum effective-sample-size ratio 90.3%    
validation   nuisance_models  2 nuisance model report(s) are available                                      
sensitivity  evalue           point=3.182, limit=2.985, source scale=mean difference; approximate conversion

Checks
------
status  count  operations                
------  -----  --------------------------
passed  2      validation.score_equations
               diagnostics.corrections   

Not run
-------
status          count  operations                       
--------------  -----  ---------------------------------
deferred        3      diagnostics.truncation_curve     
                       diagnostics.refute         

**What this output tells you.** Read the four parts in order.

| output part | what it shows on this draw |
| --- | --- |
| `Checks` and `needs attention` | `score_equations` and `corrections` pass, and no row needs attention |
| correction check | each extra equation per arm and its solved score. The contract is `theorem`, because no truncation is active |
| nuisance model diagnostics | `nuisance fits look reasonable`. The misspecified logistic propensity has a calibration slope of 0.9530 |
| reduced-regression diagnostics | the spline candidate has the lowest cross-validated risk in four of the six `gr1` fits. Each of the three families splits between its two candidates |

The nuisance report looks reasonable for an assignment model that this synthetic law makes wrong by
construction. Held-out risk compares candidates, but it cannot measure distance from the true
function. The reduced-regression table is the one place the fit shows a choice that the theorem's
conditions depend on. In most of the `gr1` fits, the data favor a nonlinear reduction. The `gr1`
and `gr2` fits serve `guard="g"`, the guard against this page's doubt.

The omitted-variable rows are `unavailable` for this fit. They address exchangeability, which DR-TMLE
does not relax. Step 10 gives the reason for the refusal.


## Step 10: sensitivity, what DR-TMLE does not relax

DR-TMLE protects the interval against one badly fitted nuisance. It does not protect against an
unmeasured confounder.
[Sensitivity analysis](../user-guide/results-assessment.md#sensitivity-analysis) asks how strong
such a confounder would need to be to change the conclusion. `cleverly` refuses the
omitted-variable bound on a DR-TMLE fit. The code prints the status of each omitted-variable
operation, and then the refusal that `robustness_value` raises.

In [10]:
from cleverly import CapabilityError

ledger = assessment.to_frame().set_index(["surface", "check"])["status"]
for operation in ("omitted_confounding", "robustness_value", "elements", "contour", "benchmark"):
    print(f"sensitivity.{operation}: {ledger.loc[('sensitivity', operation)]}")
print()
try:
    guarded.sensitivity.robustness_value()
except CapabilityError as refusal:
    bound_refusal = str(refusal)
    print("robustness value refused:", bound_refusal)
else:
    raise AssertionError("the omitted-variable bound accepted a DR-TMLE fit")

sensitivity.omitted_confounding: unavailable
sensitivity.robustness_value: unavailable
sensitivity.elements: unavailable
sensitivity.contour: unavailable
sensitivity.benchmark: unavailable

robustness value refused: sensitivity 'robustness_value' is unavailable: the omitted-variable bound has no nu^2 estimate for a 'drtmle' fit. The default estimator E[2 m(alpha_hat) - alpha_hat^2] equals nu_0^2 minus the squared error of the fitted representer, by the Riesz identity, so it falls exactly where the fitted mechanism is wrong -- the case DR-TMLE guards against. The bound would be too narrow and the robustness value too large. No derivation registered here gives nu^2, or the bound's standard error, for an estimator that does not assume a consistent treatment mechanism.


**What this output tells you.** Every omitted-variable operation is `unavailable` for this fit. The
refusal names the fitted method, `drtmle`, and the quantity the package will not estimate for it.
That quantity is $\nu^2$, the second moment of the Riesz representer.

| what the bound needs | the state on this page |
| --- | --- |
| an estimate of $\nu^2$ | the library builds the representer from the fitted assignment model, and this page doubts that model |
| a consistent assignment model | the synthetic law makes the main-effects logistic model wrong by construction |
| a derivation for the fitted estimator | the [omitted-variable bounds](../technical-reference/validation-methods.md#omitted-variable-bounds-robustness-value-benchmark-and-contours) are derived for an estimator that assumes a consistent treatment mechanism. No derivation registered here covers a DR-TMLE fit |

The refusal states the direction of the error. The default estimator of $\nu^2$ equals the true
$\nu_0^2$ minus the squared error of the fitted representer. It therefore falls exactly where the
fitted mechanism is wrong, which is the case DR-TMLE guards against. The bound would read too
narrow, and the robustness value would read too large. A number that errs toward the reassuring
answer is worse than no number, so `cleverly` refuses it here.

The `evalue` row of Step 9 still returns. It converts the reported estimate and its interval with
the outcome standard deviation, and it builds no Riesz representer. This refusal therefore does not
reach it. The row names that conversion an `approximate conversion`.

## How far to trust this

The program's question has a conditional answer. DR-TMLE reports the interval
(0.15129, 0.17801) for the ATE. The coverage of that interval rests on the rate conditions for the
outcome fit and the reduced regressions. No output on this page can verify those conditions. The
page gives no calibrated statement about hidden confounding.

| layer | establishes | does not establish |
| --- | --- | --- |
| `guard=()` equality | the variant reduces exactly to the ordinary estimator | anything about the guarded fit |
| the constant-reduction refit | that passing score checks do not separate two reductions | which reduction is adequate |
| the score and correction reports | the targeting solved all three equations, with no active truncation | the rate conditions behind the interval |
| the nuisance and reduced-regression reports | which candidate fit best under cross-validated risk | that any fitted function is consistent |
| the refused omitted-variable operations | nothing on this page, because `cleverly` computes no bound for a DR-TMLE fit | that no hidden confounder exists, or how much hidden confounding would move the estimate |

DR-TMLE ships under **conditional validity**. The registered
[canonical DR-TMLE study](../technical-reference/method-evidence/canonical-dr-tmle.md) uses a
binary complete-data law. That study has a cell for this page's case, a correct outcome regression
with a wrong assignment model. In that cell the interval cleared the coverage floor at n = 1,500,
3,000, and 6,000. At n = 1,500 the bias exceeded the equivalence margin.

No registered study covers this bounded law with flexible learners. The
[DR-TMLE evidence](../technical-reference/dr-tmle/validation-programme.md) shows where the
interval fell short of nominal coverage.


## Where to go next

If your question is which baseline variables belong in the assignment model, read
[collaborative TMLE](collaborative-tmle.ipynb). The two methods do not compose. A reduced regression
conditions on the fitted assignment mechanism as a covariate, and the C-TMLE mechanism is
deliberately not an estimate of the true one. DR-TMLE raises that refusal at fit time.

When outcomes are also missing, double robustness takes a different shape. Read
[survey non-response](survey-nonresponse.ipynb).

The [examples index](index.md#the-program) lists every tutorial in the program.
